# Step 2: Manuel Labelling with Napari

In [ ]:
import napari
from tifffile import imread
import numpy as np
import pandas as pd
from matplotlib import pyplot as plt
from matplotlib import cm
import seaborn as sns
import os
from tifffile import imwrite
import sys
sys.path.append("..")
from utility import config

def add_label_layer(viewer, label_data, layer_name):
    # Assign highlight colors
    colors = get_label_color(label_data)
    max_label = np.max(label_data)
    colors.update({max_label+1: 'red', 
                   max_label+2: 'orange', 
                   max_label+3: 'yellow', 
                   max_label+4: 'green',
                   max_label+5: 'pink',
                   max_label+6: 'blue',
                   max_label+7: 'purple',
                   max_label+8: 'brown',
                   max_label+9: 'pink',
                   max_label+10: 'cyan',
                   max_label+11: 'magenta',
                   max_label+12: 'lime',
                   max_label+13: 'olive',
                   max_label+14: 'navy',
                   max_label+15: 'teal',
                   max_label+16: 'maroon',
                   max_label+17: 'silver',
                   max_label+18: 'gold',
                   max_label+19: 'green',
                   max_label+20: 'salmon',
                   max_label+21: 'plum'})
    # Add label layer to viewer
    label_layer = viewer.add_labels(label_data, name=layer_name, opacity=0.2, colormap=colors)
    label_layer.selected_label = max_label+1
    return label_layer, max_label

def get_label_color(label):
    # Generate N distinct shades of blue using matplotlib colormap
    N = int(label.max()) + 1
    colormap_array = cm.Blues(np.linspace(0.4, 1.0, N))  # Lighter to darker blues

    # Build color dictionary for Napari
    label_color_dict = {
        i: tuple(colormap_array[i][:3])  # RGB tuple, remove alpha
        for i in range(0, N)
    }
    label_color_dict[0] = (0, 0, 0, 0)  # transparent background
    label_color_dict[None] = (0, 0, 0, 0)  # transparent background
    return label_color_dict

def save_modified_labels(label_layer, max_label, sub_id, project_dir, label_type = 'nuc'):
    # Get the modified label data
    label_data = label_layer.data
    # Within label, delete labels with id <= max_label
    label_data[label_data <= max_label] = 0
    # Relabel label to be consecutive integers starting from 1
    unique_labels = np.unique(label_data)
    label_mapping = {old_label: new_label for new_label, old_label in enumerate(unique_labels, start=1)}
    # Apply the mapping to relabel
    for old_label, new_label in label_mapping.items():
        label_data[label_data == old_label] = new_label
    # Save the modified label as a new tif file
    new_label_file_path = os.path.join(project_dir, f'sub-{sub_id:03d}', f'sub-{sub_id:03d}_data-{label_type}ManualLabel.tif')
    imwrite(new_label_file_path, label_data.astype(np.uint16))


In [ ]:
# User inputs: specify the subject ID
sub_id = 2
project_dir = config.data_dir

xyProjection_path = os.path.join(project_dir, f'sub-{sub_id:03d}', f'sub-{sub_id:03d}_data-xyProjection.tif')
nuc_label_file_path = os.path.join(project_dir, f'sub-{sub_id:03d}', f'sub-{sub_id:03d}_data-nucMask.tif')
mem_label_file_path = os.path.join(project_dir, f'sub-{sub_id:03d}', f'sub-{sub_id:03d}_data-memMask.tif')

# Load the label file
print('Loading label files...')
nuc_label = imread(nuc_label_file_path)
mem_label = imread(mem_label_file_path)
xyProjection = imread(xyProjection_path)
img_mem = xyProjection[:,0,:,:]
img_nuc = xyProjection[:,1,:,:]

# Open the label in napari
print('Opening in napari...')
viewer = napari.Viewer()
nuc_label_layer, nuc_max_label = add_label_layer(viewer, nuc_label, 'nucLabel')
mem_label_layer, mem_max_label = add_label_layer(viewer, mem_label, 'memLabel')
viewer.add_image(img_nuc, name='nucProjection', colormap='gray', blending='additive')
viewer.add_image(img_mem, name='memProjection', colormap='gray', blending='additive')

In [ ]:
print('Save modified labels after editing and closing napari viewer...')
# Close the viewer and save modified labels
viewer.close()
save_modified_labels(nuc_label_layer, nuc_max_label, sub_id, project_dir, 'nuc')
save_modified_labels(mem_label_layer, mem_max_label, sub_id, project_dir, 'mem')